In [ ]:
from pyspark.sql import SparkSession

spark=SparkSession.builder \
.appName("Milestone Assessment 1") \
.getOrCreate()


In [ ]:
%%writefile customers.csv
customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active

In [ ]:
%%writefile usage.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210

In [ ]:
%%writefile plans.json
[
{
"plan_id": "P101",
"plan_name": "Smart Basic",
"monthly_fee": 499,
"data_limit_gb": 50,
"features": {
"unlimited_calls": true,
"ott_included": false,
"roaming": "National"
}
},
{

"plan_id": "P102",
"plan_name": "Smart Plus",
"monthly_fee": 799,
"data_limit_gb": 75,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "National"
}
},
{
"plan_id": "P103",
"plan_name": "Budget Saver",
"monthly_fee": 299,
"data_limit_gb": 25,
"features": {
"unlimited_calls": false,
"ott_included": false,
"roaming": null
}
},
{
"plan_id": "P104",
"plan_name": "Premium Max",
"monthly_fee": 1199,
"data_limit_gb": 100,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "International"
}
}
]

In [ ]:
%%writefile payments.csv
payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Telecom Analytics").getOrCreate()


In [ ]:
#1
customers_df = spark.read.option("header", True).option("inferSchema", True).csv("customers.csv")

In [ ]:
#2
usage_df = spark.read.option("header", True).option("inferSchema", True).csv("usage.csv")

In [ ]:
#3
payments_df = spark.read.option("header", True).option("inferSchema", True).csv("payments.csv")

In [ ]:
#4
plans_df = spark.read.option("multiline", True).option("inferSchema", True).json("plans.json")

In [ ]:
#5
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_df.printSchema()

In [ ]:
#6
print("Customers:", customers_df.count())
print("Usage:", usage_df.count())
print("Payments:", payments_df.count())
print("Plans:", plans_df.count())

In [ ]:
#7
customers_df.write.mode("overwrite").parquet("bronze/customers")
usage_df.write.mode("overwrite").parquet("bronze/usage")
payments_df.write.mode("overwrite").parquet("bronze/payments")
plans_df.write.mode("overwrite").parquet("bronze/plans")

In [ ]:
#8
customers_df.filter(col("plan_id").isNull()).show()

In [ ]:
#9
usage_df.filter(col("data_used_gb").isNull()).show()

In [ ]:
#10
payments_df.filter(
    col("amount_paid").isNull()
).show()

In [ ]:
#11
payments_df.filter(
    col("payment_mode").isNull()
).show()

In [ ]:
#12
usage_df = usage_df.fillna(
    {"data_used_gb": 0}
)
usage_df.show()

In [ ]:
#13
payments_df = payments_df.fillna(
    {"amount_paid": 0}
)

payments_df.show()

In [ ]:
#14
payments_df = payments_df.fillna(
    {"payment_mode": "Not Provided"}
)

payments_df.show()

In [ ]:
#15
customers_df = customers_df.fillna(
    {"plan_id": "UNKNOWN"}
)
customers_df.show()

In [ ]:
#16
customers_df = customers_df.withColumn(
    "data_quality_status",
    when(
        col("plan_id") == "UNKNOWN",
        "Review"
    ).otherwise("Valid")
)

customers_df.show()

In [ ]:
#17
customers_df.write.mode("overwrite").parquet("silver/customers")
usage_df.write.mode("overwrite").parquet("silver/usage")
payments_df.write.mode("overwrite").parquet("silver/payments")

In [ ]:
#18
plans_flat_df = plans_df.select(
    col("plan_id"),
    col("plan_name"),
    col("monthly_fee"),
    col("data_limit_gb"),
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming")
)

plans_flat_df.show()

In [ ]:
#19
plans_df.select(
    col("plan_id"),
    col("features.unlimited_calls").alias("unlimited_calls")
).show()

In [ ]:
#20
plans_df.select(
    col("plan_id"),
    col("features.ott_included").alias("ott_included")
).show()

In [ ]:
#21
plans_df.select(
    col("plan_id"),
    col("features.roaming").alias("roaming")
).show()

In [ ]:
#22
plans_flat_df = plans_flat_df.withColumn(
    "roaming",
    when(
        col("roaming").isNull(),
        "Not Available"
    ).otherwise(col("roaming"))
)

plans_flat_df.show()

In [ ]:
#23
plans_flat_df.write.mode("overwrite").parquet("silver/plans")

In [ ]:
#24
customer_plan_df = customers_df.join(
    plans_flat_df,
    "plan_id",
    "left"
)

customer_plan_df.show()

In [ ]:
#25
customer_usage_df = customers_df.join(
    usage_df,
    "customer_id",
    "left"
)

customer_usage_df.show()

In [ ]:
#26
customer_payment_df = customers_df.join(
    payments_df,
    "customer_id",
    "left"
)

customer_payment_df.show()

In [ ]:
#27
complete_df = customers_df \
    .join(plans_flat_df, "plan_id", "left") \
    .join(usage_df, "customer_id", "left") \
    .join(payments_df, "customer_id", "left")

complete_df.show()

In [ ]:
#28
customers_df.join(
    plans_flat_df,
    "plan_id",
    "left_anti"
).show()

In [ ]:
#29
usage_df.join(
    customers_df,
    "customer_id",
    "left_anti"
).show()

In [ ]:
#30
payments_df.join(
    customers_df,
    "customer_id",
    "left_anti"
).show()

In [ ]:
#31
complete_df = complete_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Low User")
)

complete_df.show()

In [ ]:
#32
complete_df = complete_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000, "High Payment")
    .when(col("amount_paid") >= 500, "Medium Payment")
    .otherwise("Low Payment")
)

complete_df.show()

In [ ]:
#33
complete_df = complete_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(col("data_used_gb") < 15, "Medium Risk")
    .otherwise("Low Risk")
)

complete_df.show()

In [ ]:
#34
complete_df = complete_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)

complete_df.show()

In [ ]:
#35
complete_df = complete_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes")
    .otherwise("No")
)

complete_df.show()

In [ ]:
#36
complete_df.groupBy("city").count().show()

In [ ]:
#37
complete_df.groupBy("state").count().show()

In [ ]:
#38
complete_df.groupBy("plan_id").count().show()

In [ ]:
#39
complete_df.groupBy("usage_category").count().show()

In [ ]:
#40
complete_df.groupBy("churn_risk").count().show()

In [ ]:
#41
complete_df.groupBy("plan_name") \
    .agg(
        sum("data_used_gb").alias("total_data_usage")
    ) \
    .show()

In [ ]:
#42
complete_df.groupBy("plan_name") \
    .agg(
        avg("data_used_gb").alias("average_data_usage")
    ) \
    .show()

In [ ]:
#43
complete_df.groupBy("city") \
    .agg(
        sum("call_minutes").alias("total_call_minutes")
    ) \
    .show()

In [ ]:
#44
complete_df.groupBy("state") \
    .agg(
        sum("sms_count").alias("total_sms")
    ) \
    .show()

In [ ]:
#45
complete_df.filter(
    col("payment_status") == "Success"
).agg(
    sum("amount_paid").alias("total_revenue")
).show()

In [ ]:
#46
complete_df.groupBy("city") \
    .agg(
        sum("amount_paid").alias("city_revenue")
    ) \
    .show()

In [ ]:
#47
complete_df.groupBy("plan_name") \
    .agg(
        sum("amount_paid").alias("plan_revenue")
    ) \
    .show()

In [ ]:
#48
complete_df.groupBy("payment_mode") \
    .agg(
        sum("amount_paid").alias("payment_mode_revenue")
    ) \
    .show()

In [ ]:
#49
complete_df.groupBy("plan_name") \
    .agg(
        sum("amount_paid").alias("total_revenue")
    ) \
    .orderBy(
        col("total_revenue").desc()
    ) \
    .show(1)

In [ ]:
#50
complete_df.groupBy("city") \
    .agg(
        sum("amount_paid").alias("total_revenue")
    ) \
    .orderBy(
        col("total_revenue").desc()
    ) \
    .show(1)

In [ ]:
#51
window_spec = Window.orderBy(
    col("data_used_gb").desc()
)

complete_df.withColumn(
    "data_rank",
    rank().over(window_spec)
).show()

In [ ]:
#52
window_spec = Window.orderBy(
    col("amount_paid").desc()
)

complete_df.withColumn(
    "payment_rank",
    rank().over(window_spec)
).show()

In [ ]:
#53
window_spec = Window.orderBy(
    col("data_used_gb").desc()
)

complete_df.withColumn(
    "data_rank",
    rank().over(window_spec)
).filter(
    col("data_rank") <= 3
).show()

In [ ]:
#54
window_spec = Window.orderBy(
    col("amount_paid").desc()
)

complete_df.withColumn(
    "revenue_rank",
    rank().over(window_spec)
).filter(
    col("revenue_rank") <= 3
).show()

In [ ]:
#55
window_spec = Window.partitionBy(
    "city"
).orderBy(
    col("data_used_gb").desc()
)

complete_df.withColumn(
    "city_rank",
    row_number().over(window_spec)
).filter(
    col("city_rank") == 1
).show()

In [ ]:
#56
window_spec = Window.partitionBy(
    "plan_name"
).orderBy(
    col("data_used_gb").desc()
)

complete_df.withColumn(
    "plan_rank",
    row_number().over(window_spec)
).filter(
    col("plan_rank") == 1
).show()

In [ ]:
#57
monthly_revenue = complete_df.groupBy(
    "bill_month"
).agg(
    sum("amount_paid").alias("monthly_revenue")
)

window_spec = Window.orderBy("bill_month")

monthly_revenue.withColumn(
    "running_total_revenue",
    sum("monthly_revenue").over(window_spec)
).show()

In [ ]:
#58
window_spec = Window.partitionBy(
    "customer_id"
).orderBy(
    "usage_month"
)

complete_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb").over(window_spec)
).show()

In [ ]:
#59
window_spec = Window.partitionBy(
    "customer_id"
).orderBy(
    "usage_month"
)

complete_df.withColumn(
    "next_month_usage",
    lead("data_used_gb").over(window_spec)
).show()

In [ ]:
#60
window_spec = Window.partitionBy(
    "customer_id"
).orderBy(
    "usage_month"
)

complete_df.withColumn(
    "previous_usage",
    lag("data_used_gb").over(window_spec)
).filter(
    col("data_used_gb") > col("previous_usage")
).show()

In [ ]:
#61
customers_df.createOrReplaceTempView("customers")
usage_df.createOrReplaceTempView("usage")
payments_df.createOrReplaceTempView("payments")
plans_flat_df.createOrReplaceTempView("plans")
complete_df.createOrReplaceTempView("complete_view")

In [ ]:
#62
spark.sql("""
select *
from customers
where status='Active'
""").show()

In [ ]:
#63
spark.sql("""
select city,
count(*) as total_customers
from customers
group by city
""").show()

In [ ]:
#64
spark.sql("""
select p.plan_name,
sum(pm.amount_paid) as revenue
from customers c
join plans p
on c.plan_id=p.plan_id
join payments pm
on c.customer_id=pm.customer_id
group by p.plan_name
""").show()

In [ ]:
#65
spark.sql("""
select *
from complete_view
where usage_category='Heavy User'
""").show()

In [ ]:
#66
spark.sql("""
select *
from complete_view
where churn_risk='High Risk'
""").show()

In [ ]:
#67
spark.sql("""
select *
from customers
where plan_id='UNKNOWN'
""").show()

In [ ]:
#68
spark.sql("""
select *
from payments
where payment_status in ('Failed','Pending')
""").show()

In [ ]:
#69
spark.sql("""
select *
from complete_view
order by data_used_gb desc
limit 5
""").show()

In [ ]:
#70
spark.sql("""
select payment_mode,
sum(amount_paid) as revenue
from payments
group by payment_mode
""").show()

In [ ]:
#71
gold_df = complete_df

gold_df.write.mode("overwrite").parquet("gold/customer_gold")

In [ ]:
#72
complete_df.write \
.mode("overwrite") \
.partitionBy("usage_month") \
.parquet("gold/customer_gold")

In [ ]:
#73
incremental_df = spark.createDataFrame(
[
(101, "2026-03", 55, 1100, 150),
(102, "2026-03", 40, 750, 100),
(104, "2026-03", 62, 1300, 180)
],
[
"customer_id",
"usage_month",
"data_used_gb",
"call_minutes",
"sms_count"
]
)

incremental_df.show()

In [ ]:
#74
incremental_usage_df = spark.read \
    .option("header", True) \
    .csv("usage.csv")

In [ ]:
#75
updated_usage_df = usage_df.select(
    col("usage_id").cast("int").alias("usage_id"),
    col("customer_id").cast("int").alias("customer_id"),
    col("usage_month"),
    col("data_used_gb").cast("int").alias("data_used_gb"),
    col("call_minutes").cast("int").alias("call_minutes"),
    col("sms_count").cast("int").alias("sms_count")
).unionByName(
    incremental_usage_df.select(
        col("usage_id").cast("int").alias("usage_id"),
        col("customer_id").cast("int").alias("customer_id"),
        col("usage_month"),
        col("data_used_gb").cast("int").alias("data_used_gb"),
        col("call_minutes").cast("int").alias("call_minutes"),
        col("sms_count").cast("int").alias("sms_count")
    )
)

updated_usage_df.show()

In [ ]:
#76
updated_usage_df.groupBy(
    "customer_id"
).agg(
    sum("data_used_gb").alias("total_data_used"),
    sum("call_minutes").alias("total_call_minutes"),
    sum("sms_count").alias("total_sms")
).show()

In [ ]:
#77
updated_usage_df.write.mode("overwrite").parquet("silver/usage_fixed")

In [ ]:
#78
before_count = usage_df.count()
after_count = updated_usage_df.count()

print("Before Load:", before_count)
print("After Load:", after_count)
print("New Records Added:", after_count - before_count)

In [ ]:
#79
customer_usage_summary = gold_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_flag",
    "amount_paid",
    "payment_status",
    "churn_risk"
)

customer_usage_summary.show()

In [ ]:
#80
plan_performance_report = gold_df.groupBy(
    "plan_name"
).agg(
    count("customer_id").alias("total_customers"),
    sum("data_used_gb").alias("total_data_usage"),
    avg("data_used_gb").alias("average_data_usage"),
    sum("amount_paid").alias("total_revenue")
)

plan_performance_report.show()

In [ ]:
#81
city_revenue_report = gold_df.groupBy(
    "city"
).agg(
    count("customer_id").alias("total_customers"),
    sum("amount_paid").alias("total_revenue"),
    avg("amount_paid").alias("average_payment")
)

city_revenue_report.show()

In [ ]:
#82
churn_risk_report = gold_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "status",
    "churn_risk"
)

churn_risk_report.show()

In [ ]:
#83
over_usage_report = gold_df.select(
    "customer_id",
    "customer_name",
    "plan_name",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
)

over_usage_report.show()